In [1]:
import os
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from xgboost import XGBClassifier

In [2]:
np.random.seed(42)

damage_classes = [
    "Front Breakage",
    "Front Crushed",
    "Rear Breakage",
    "Rear Crushed"
]

base_cost = {
    "Front Breakage": 40000,
    "Front Crushed": 90000,
    "Rear Breakage": 35000,
    "Rear Crushed": 80000
}

n_samples = 3000
rows = []

for _ in range(n_samples):
    damage = np.random.choice(damage_classes)
    vehicle_age = np.random.randint(0, 15)
    mileage = np.random.randint(0, 200000)

    repair_cost = (
        base_cost[damage]
        + vehicle_age * 1200
        + mileage * 0.05
        + np.random.normal(0, 5000)
    )

    repair_cost = max(1000, repair_cost)

    # Assign severity labels
    if repair_cost < 25000:
        severity = "Minor"
    elif repair_cost < 60000:
        severity = "Moderate"
    elif repair_cost < 150000:
        severity = "Severe"
    else:
        severity = "Total Loss"

    rows.append([
        damage,
        vehicle_age,
        mileage,
        round(repair_cost, 2),
        severity
    ])

df = pd.DataFrame(
    rows,
    columns=[
        "damage_class",
        "vehicle_age",
        "mileage",
        "repair_cost",
        "claim_severity"
    ]
)

df.head()

,damage_class,vehicle_age,mileage,repair_cost,claim_severity
0,Rear Breakage,3,131932,47915.32,Moderate
1,Front Breakage,6,137337,50989.70,Moderate
2,Rear Breakage,6,168266,40558.49,Moderate
3,Rear Crushed,7,41090,87990.48,Severe
4,Front Crushed,4,769,96003.93,Severe


In [3]:
os.makedirs("../data/synthetic", exist_ok=True)

df.to_csv(
    "../data/synthetic/claim_severity_data.csv",
    index=False
)

print("Dataset saved successfully.")

Dataset saved successfully.


In [4]:
print(df.shape)
print(df["claim_severity"].value_counts())

(3000, 5)
claim_severity
Severe      1705
Moderate    1295
Name: count, dtype: int64


In [5]:
X = df[
    [
        "damage_class",
        "vehicle_age",
        "mileage",
        "repair_cost"
    ]
]

y = df["claim_severity"]

In [6]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print(label_encoder.classes_)

['Moderate' 'Severe']


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

In [8]:
categorical_features = ["damage_class"]
numeric_features = [
    "vehicle_age",
    "mileage",
    "repair_cost"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "num",
            "passthrough",
            numeric_features
        )
    ]
)

In [9]:
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            XGBClassifier(
                n_estimators=200,
                max_depth=6,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=42,
                eval_metric="mlogloss"
            )
        )
    ]
)

In [10]:
model.fit(X_train, y_train)

print("Model training completed.")

Model training completed.


In [11]:
y_pred = model.predict(X_test)

In [12]:
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=label_encoder.classes_
    )
)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.9983

Classification Report:
              precision    recall  f1-score   support

    Moderate       1.00      1.00      1.00       259
      Severe       1.00      1.00      1.00       341

    accuracy                           1.00       600
   macro avg       1.00      1.00      1.00       600
weighted avg       1.00      1.00      1.00       600


Confusion Matrix:
[[258   1]
 [  0 341]]


In [13]:
os.makedirs("../saved_models", exist_ok=True)

joblib.dump(
    model,
    "../saved_models/severity_model.pkl"
)

joblib.dump(
    label_encoder,
    "../saved_models/severity_label_encoder.pkl"
)

print("Model and label encoder saved successfully.")

Model and label encoder saved successfully.


In [14]:
sample = pd.DataFrame({
    "damage_class": ["Front Crushed"],
    "vehicle_age": [5],
    "mileage": [70000],
    "repair_cost": [105000]
})

pred_encoded = model.predict(sample)[0]
pred_label = label_encoder.inverse_transform([pred_encoded])[0]

print("Predicted Claim Severity:", pred_label)

Predicted Claim Severity: Severe


In [15]:
loaded_model = joblib.load(
    "../saved_models/severity_model.pkl"
)

loaded_encoder = joblib.load(
    "../saved_models/severity_label_encoder.pkl"
)

pred = loaded_model.predict(sample)[0]
severity = loaded_encoder.inverse_transform([pred])[0]

print("Loaded Model Prediction:", severity)

Loaded Model Prediction: Severe
